<a href="https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#Baseline Rule

Pages with high impressions but low CTR receive the highest action score because they have the greatest opportunity to gain additional clicks through optimization.

### Reason Codes

- **RC1:** High impressions and low CTR – highest priority for optimization.
- **RC2:** Medium impressions and low CTR – moderate priority.
- **RC3:** High visibility but below-average CTR – investigate title, meta description, or content quality.
- **RC4:** Low impressions – lower priority because optimization is likely to have a smaller impact.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# Build the ranked queue

A baseline Action Score is calculated using impressions and CTR. Pages with high impressions and low CTR receive a higher score because improving these pages could generate more additional clicks. The pages are ranked by Action Score, and the ranked queue is saved as `work/outputs/baseline_action_score.csv`.

In [3]:
!pip install -q duckdb datasets huggingface_hub   #connecting the dataset- install the required libraries

In [18]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")


Token loaded successfully!
Connected successfully!


In [19]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [20]:
baseline = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [21]:
baseline["ctr"] = (
    baseline["gsc_clicks"] /
    baseline["gsc_impressions"]
)

In [23]:
baseline["action_score"] = (
    baseline["gsc_impressions"] *
    (1 - baseline["ctr"])
)

In [24]:
baseline = baseline.sort_values(
    by="action_score",
    ascending=False
)

top20 = baseline.head(20).copy()

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
top20 = baseline.head(20).copy()

In [26]:
top20["action"] = "Optimize page"              #Action

In [27]:
top20["reason_code"] = "RC1"                   #Reason_Code

In [28]:
top20["confidence_note"] = (                                        #confidence_note
    "Moderate confidence - based on baseline rule only"
)

In [29]:
top20["what_could_make_it_wrong"] = (
    "Daily fluctuations or seasonal effects may reduce the reliability of this recommendation."
)

In [30]:
top20[
    [
        "content_hash_id",
        "action_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_could_make_it_wrong"
    ]
]

,content_hash_id,action_score,action,reason_code,confidence_note,what_could_make_it_wrong
3534021,content_44f34c0a90047651,40083.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
2949804,content_eadb33b5df496f4a,39053.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
43030,content_34a70fea29d15f24,39001.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
3358182,content_eadb33b5df496f4a,38165.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
43077,content_945d6ff91386c817,37368.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
3274261,content_eadb33b5df496f4a,35179.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
3169375,content_eadb33b5df496f4a,34594.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
3578638,content_eadb33b5df496f4a,34371.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
3248019,content_fec55986a1868d62,33383.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...
2677007,content_eadb33b5df496f4a,33356.0,Optimize page,RC1,Moderate confidence - based on baseline rule only,Daily fluctuations or seasonal effects may red...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

Some recommendations may not be ideal because the same content item can appear on multiple report dates, resulting in repeated entries in the ranked queue. In a production system, these daily records could be aggregated into a single recommendation per content item.

No product flags, future information, or label-derived features were used to calculate the Action Score. The baseline uses only current-month impressions and CTR, making it suitable as an explainable decision-support baseline.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20["content_hash_id"].value_counts()

,count
content_hash_id,
content_eadb33b5df496f4a,10
content_44f34c0a90047651,6
content_fec55986a1868d62,2
content_34a70fea29d15f24,1
content_945d6ff91386c817,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.